In [4]:
!pip install -q aerosandbox robotframework

In [ ]:
import aerosandbox.geometry as ag

# Define parameters for the biplane wings
span = 10.0       # Total wingspan for each wing (symmetric)
chord = 1.0       # Chord length for the rectangular wings
separation = 2.0  # Vertical separation between the lower and upper wing (z-axis)
stagger = 0.5     # Horizontal stagger (x-axis offset) of the upper wing relative to the lower wing
airfoil = ag.Airfoil("naca0012") # Define a simple airfoil shape

# --- Create the Lower Wing ---
# Define the cross-sections for the lower wing.
# For a rectangular wing, we need two cross-sections: root and tip.
# xyz_le is the [x, y, z] coordinate of the leading edge.
lower_wing_xsecs = [
    ag.WingXSec(
        xyz_le=[0, 0, 0],      # Root at the origin [x, y, z]
        chord=chord,
        airfoil=airfoil
    ),
    ag.WingXSec(
        xyz_le=[0, span / 2, 0], # Tip at half span in y, same x and z as root
        chord=chord,
        airfoil=airfoil
    )
]

# Create the Wing object for the lower wing.
# symmetric=True means the defined half-span is mirrored across the XZ plane (y=0).
lower_wing = ag.Wing(
    name="Lower Wing",
    xsecs=lower_wing_xsecs,
    symmetric=True
)

# --- Create the Upper Wing ---
# Define the cross-sections for the upper wing.
# These are similar to the lower wing but offset in x (stagger) and z (separation).
upper_wing_xsecs = [
    ag.WingXSec(
        xyz_le=[stagger, 0, separation],      # Root offset by stagger in x and separation in z
        chord=chord,
        airfoil=airfoil
    ),
    ag.WingXSec(
        xyz_le=[stagger, span / 2, separation], # Tip offset by stagger in x and separation in z, half span in y
        chord=chord,
        airfoil=airfoil
    )
]

# Create the Wing object for the upper wing, also symmetric.
upper_wing = ag.Wing(
    name="Upper Wing",
    xsecs=upper_wing_xsecs,
    symmetric=True
)

# --- Create the Airplane ---
# Create the Airplane object, which is a container for the wings (and other components).
biplane = ag.Airplane(
    name="Simple Biplane",
    wings=[lower_wing, upper_wing] # Add both created wings to the airplane
)

In [4]:
biplane

Airplane 'Simple Biplane' (2 wings, 0 fuselages)

In [5]:
import os
from robot.libdoc import LibraryDocumentation

# Define a placeholder path for a resource file.
# In a real application, this path should point to your actual .robot resource file.
# For this example, we'll assume a file exists at this location.
# You might need to create a dummy file for the code to run without error.
# Example dummy resource file content (e.g., in 'my_resource.robot'):
# *** Keywords ***
# My First Keyword
#     Log    Hello from first keyword
#
# Another Keyword
#     Log    Hello from another keyword
#
resource_file_path = r'C:\Users\preet\Documents\gurveer_stuff\kmit-major-project\CodeSage\my_keywords.resource'

# --- Create a dummy resource file for demonstration if it doesn't exist ---
# This part is just to make the example runnable.
# In a real scenario, you would use your existing resource file.
if not os.path.exists(resource_file_path):
    print(f"Creating a dummy resource file: {resource_file_path}")
    dummy_content = """
*** Keywords ***
My First Keyword
    Log    This is the first keyword

Another Keyword With Args
    [Arguments]    ${arg1}    ${arg2}
    Log    Args: ${arg1}, ${arg2}

Keyword Without Steps
    [Documentation]    This keyword has no steps.
"""
    with open(resource_file_path, 'w') as f:
        f.write(dummy_content.strip())
    print("Dummy file created.")
# --- End of dummy file creation ---

try:
    # Use LibraryDocumentation to parse the resource file
    # The first argument is the path to the resource file
    lib_doc = LibraryDocumentation(resource_file_path)

    print(f"Successfully processed resource file: {resource_file_path}")
    print(f"Resource Name (from file): {lib_doc.name}") # LibraryDoc also has a name attribute

    # Access the keywords attribute which is a list of keyword objects
    keywords = lib_doc.keywords

    if keywords:
        print("\nKeywords found:")
        # Iterate through the keyword objects and print their names
        for kw in keywords:
            print(f"- {kw.name}")
    else:
        print("\nNo keywords found in the resource file.")

except Exception as e:
    print(f"An error occurred while processing the resource file: {e}")
    print(f"Please ensure '{resource_file_path}' is a valid Robot Framework resource file.")

finally:
    # --- Clean up the dummy file ---
    # This part is just for the example's cleanup.
    if os.path.exists(resource_file_path) and "Creating a dummy resource file" in locals().get('dummy_content', ''):
         print(f"\nCleaning up dummy resource file: {resource_file_path}")
         os.remove(resource_file_path)
    # --- End of cleanup ---


Successfully processed resource file: C:\Users\preet\Documents\gurveer_stuff\kmit-major-project\CodeSage\my_keywords.resource
Resource Name (from file): my_keywords

Keywords found:
- Calculate Sum
- Log Message
- Verify Text Contains


In [5]:
from robot.api import get_resource_from_file, ResourceFile
from robot.libdocpkg.model import KeywordDoc


def extract_keywords_from_resource(resource_file_path: str) -> list[KeywordDoc]:
    """
    Extracts all keywords from a Robot Framework resource file, including their
    documentation and arguments.

    Args:
        resource_file_path: The path to the Robot Framework resource file (.resource).

    Returns:
        A list of KeywordDoc objects, each representing a keyword found in the resource file.
        Returns an empty list if the resource file is invalid or no keywords are found.
    """
    try:
        resource: ResourceFile = get_resource_from_file(resource_file_path)
    except Exception as e:
        print(f"Error parsing resource file: {e}")
        return []

    keywords: list[KeywordDoc] = resource.keywords

    return keywords


def print_keyword_details(keywords: list[KeywordDoc]) -> None:
    """
    Prints the details of each keyword extracted from a resource file.

    Args:
        keywords: A list of KeywordDoc objects.
    """
    if not keywords:
        print("No keywords found.")
        return

    for keyword in keywords:
        print("-" * 40)
        print(f"Keyword Name: {keyword.name}")
        print(f"Documentation: {keyword.doc}")
        print(f"Arguments: {keyword.args}")
        print(f"Source: {keyword.source}:{keyword.lineno}")
        print("-" * 40)


if __name__ == "__main__":
    # Replace 'path/to/your/resource.resource' with the actual path to your resource file
    resource_file_path = r'C:\Users\preet\Documents\gurveer_stuff\kmit-major-project\CodeSage\my_keywords.resource'

    # Sample Resource file Creation for demo purpose (my_resource.resource)
    with open(resource_file_path, 'w') as f:
        f.write("""***Settings***
Documentation    Example resource file for keyword extraction.

***Keywords***
Log Message
    [Arguments]    ${message}
    Log    ${message}

Multiply Numbers
    [Arguments]    ${num1}    ${num2}
    ${result} =    Evaluate    ${num1} * ${num2}
    [Return]    ${result}

No Argument Keyword
    Log    This keyword has no arguments.
""")

    keywords = extract_keywords_from_resource(resource_file_path)
    print_keyword_details(keywords)

ImportError: cannot import name 'get_resource_from_file' from 'robot.api' (c:\Users\preet\Documents\gurveer_stuff\kmit-major-project\.venv\Lib\site-packages\robot\api\__init__.py)